In [303]:
import pandas as pd
from pathlib import Path

data_path = Path("data")
data_path.mkdir(exist_ok=True)

df = pd.read_csv(data_path / "data_combined.csv")
df.head()

,date,temp_min,temp_max,temp_mean,city
0,1996-05-18,6.6,8.5,7.700000,Malmö
1,1996-05-19,8.5,8.9,8.700000,Malmö
2,1996-05-20,7.4,8.4,7.800000,Malmö
3,1996-05-21,6.5,7.7,7.166667,Malmö
4,1996-05-22,6.5,11.4,9.700000,Malmö


In [304]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['city'] = le.fit_transform(df['city'])
df["date"] = pd.to_datetime(df["date"])
df['day_of_year'] = pd.to_datetime(df['date']).dt.day_of_year
df['month']       = pd.to_datetime(df['date']).dt.month
df['year']        = pd.to_datetime(df['date']).dt.year

df = df.sort_values('date').reset_index(drop=True)

In [305]:
print(df["date"].dtype)

datetime64[us]


In [306]:
df.columns

Index(['date', 'temp_min', 'temp_max', 'temp_mean', 'city', 'day_of_year',
       'month', 'year'],
      dtype='str')

In [307]:
df["city"].unique()

array([2, 0, 1])

In [308]:
le.classes_

array(['Malmö', 'Stockholm', 'Vinga A'], dtype=object)

In [309]:
from sklearn.ensemble import GradientBoostingRegressor

X = df[['city', 'day_of_year', 'month', 'year']]
y = df['temp_mean']

split = int(len(df) * 0.8)

X_train = X.iloc[:split]   # older 80% of data
X_test  = X.iloc[split:]   # newer 20% of data

y_train = y.iloc[:split]
y_test  = y.iloc[split:]

model = GradientBoostingRegressor()
model.fit(X_train, y_train)

,"loss loss: {'squared_error', 'absolute_error', 'huber', 'quantile'}, default='squared_error'Loss function to be optimized. 'squared_error' refers to the squarederror for regression. 'absolute_error' refers to the absolute error ofregression and is a robust loss function. 'huber' is acombination of the two. 'quantile' allows quantile regression (use`alpha` to specify the quantile).See:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_quantile.py`for an example that demonstrates quantile regression for creatingprediction intervals with `loss='quantile'`.",'squared_error'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.",0.1
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",100
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'The function to measure the quality of a split. Supported criteria are""friedman_mse"" for the mean squared error with improvement score byFriedman, ""squared_error"" for mean squared error. The default value of""friedman_mse"" is generally the best as it can provide a betterapproximation in some cases... versionadded:: 0.18",'friedman_mse'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft 

In [310]:
len(X_train), len(X_test), len(y_train), len(y_test)

(22768, 5692, 22768, 5692)

In [311]:
from sklearn.metrics import mean_squared_error

y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)

mse

11.13522708811748

In [312]:
def make_input_data(date: str):
    user_date = pd.to_datetime(date)

    user_input = pd.DataFrame({
        'city': [3],
        'day_of_year': [user_date.day_of_year],
        'month':       [user_date.month],
        'year':        [user_date.year]
    })
    return user_input

In [313]:
import pandas as pd

user_input = make_input_data("2022-01-16")

predicted_temp = model.predict(user_input)
print(f"Predicted temp_mean: {predicted_temp[0]:.1f} °C")

Predicted temp_mean: 4.4 °C


In [314]:
model_low  = GradientBoostingRegressor(loss='quantile', alpha=0.1)  # 10th percentile
model_mid  = GradientBoostingRegressor(loss='quantile', alpha=0.5)  # median
model_high = GradientBoostingRegressor(loss='quantile', alpha=0.9)  # 90th percentile

model_low.fit(X_train, y_train)
model_mid.fit(X_train, y_train)
model_high.fit(X_train, y_train)

# Predict
low  = model_low.predict(user_input)[0]
mid  = model_mid.predict(user_input)[0]
high = model_high.predict(user_input)[0]

print(f"Predicted: {mid:.1f} °C  (range: {low:.1f} – {high:.1f} °C)")
print(f"+- {round((high - low) / 2)}")

Predicted: 5.3 °C  (range: 4.2 – 6.3 °C)
+- 1


In [315]:
import joblib

models_path = Path("models")
models_path.mkdir(exist_ok=True)

# Save
models = {
    'low':  model_low,
    'mid':  model_mid,
    'high': model_high
}
joblib.dump(models, models_path / 'temperature_models.pkl')

['models/temperature_models.pkl']

In [316]:
models = joblib.load(models_path / 'temperature_models.pkl')
predicted = models['mid'].predict(user_input)
predicted[0]

np.float64(5.259989336414284)

In [317]:
from xgboost import XGBRegressor

xgboost = XGBRegressor(
    n_estimators=500, 
    learning_rate=0.05,   
    max_depth=6,           
    subsample=0.8,         
    colsample_bytree=0.8,  
    random_state=42
)

xgboost

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [318]:
xgboost.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [319]:
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import numpy as np

y_pred = xgboost.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

mse, rmse, r2

(11.13522708811748, np.float64(3.6333450436031285), 0.7400776081974858)

In [320]:
predicted_temp = xgboost.predict(user_input)[0]
print(f"Predicted temp_mean: {predicted_temp:.1f} °C")

Predicted temp_mean: 6.0 °C


In [321]:
model_low  = XGBRegressor(objective='reg:quantileerror', quantile_alpha=0.1)
model_mid  = XGBRegressor(objective='reg:quantileerror', quantile_alpha=0.5)
model_high = XGBRegressor(objective='reg:quantileerror', quantile_alpha=0.9)

model_low.fit(X_train, y_train)
model_mid.fit(X_train, y_train)
model_high.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:quantileerror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabete

In [322]:
X_test

,city,day_of_year,month,year
22768,0,103,4,2020
22769,1,103,4,2020
22770,0,104,4,2020
22771,1,104,4,2020
22772,2,104,4,2020
...,...,...,...,...
28455,0,30,1,2026
28456,2,30,1,2026
28457,2,31,1,2026
28458,0,31,1,2026


In [323]:
y_test

22768    11.25
22769     7.85
22770     5.30
22771     3.65
22772     5.85
         ...  
28455    -1.30
28456    -2.20
28457    -1.45
28458    -2.80
28459    -5.00
Name: temp_mean, Length: 5692, dtype: float64

In [324]:
user_input = make_input_data("2022-01-15")

low = model_low.predict(user_input)[0]
mid = model_mid.predict(user_input)[0]
high = model_high.predict(user_input)[0]

print(f"Predicted: {mid:.1f} °C  (range: {low:.1f} – {high:.1f} °C)")
margin = round((high - low) / 2, 1)
print(f"±{margin:.1f}°C")

Predicted: 6.1 °C  (range: 5.8 – 8.1 °C)
±1.1°C


In [325]:
low_y_pred = model_low.predict(X_test)
mid_y_pred = model_mid.predict(X_test)
high_y_pred = model_high.predict(X_test)

low_mse = mean_squared_error(y_test, low_y_pred)
low_mae  = mean_absolute_error(y_test, low_y_pred)
low_rmse = np.sqrt(mean_squared_error(y_test, low_y_pred))
low_r2   = r2_score(y_test, low_y_pred)

mid_mse = mean_squared_error(y_test, mid_y_pred)
mid_mae  = mean_absolute_error(y_test, mid_y_pred)
mid_rmse = np.sqrt(mean_squared_error(y_test, mid_y_pred))
mid_r2   = r2_score(y_test, mid_y_pred)

high_mse = mean_squared_error(y_test, high_y_pred)
high_mae  = mean_absolute_error(y_test, high_y_pred)
high_rmse = np.sqrt(mean_squared_error(y_test, high_y_pred))
high_r2   = r2_score(y_test, high_y_pred)

low_mse, low_mae, low_rmse, low_r2

(12.29177508774409,
 2.672256756777,
 np.float64(3.5059627904106585),
 0.7579834788848109)

In [326]:
mid_mse, mid_mae, mid_rmse, mid_r2

(14.630773259728338,
 2.9435158729678883,
 np.float64(3.825019380307548),
 0.7119302281185453)

In [327]:
high_mse, high_mae, high_rmse, high_r2

(20.543689880196286,
 3.525371565185461,
 np.float64(4.532514741310423),
 0.5955090033634095)

In [328]:
models = {
    'low':  model_low,
    'mid':  model_mid,
    'high': model_high
}
joblib.dump(models, models_path / 'temperature_models.pkl')

['models/temperature_models.pkl']